# M04 · Performance & Scaling — Hands-On Exercise
## ShopMax: Making Maxi Ready for the Diwali Mega Sale

**Time:** 60 minutes
**Name:** ___________________________

---

### Setup — run this first, do not modify

The cell below loads the mock LLM and helpers. Run it once, then move to Exercise 1.

In [1]:
import time, random, numpy as np, asyncio, operator, concurrent.futures
from typing import TypedDict, Annotated, List, Optional

# ── Mock LLM — no API key needed ─────────────────────────────────────────────
class MockResponse:
    def __init__(self, content, model='gpt-4o-mini', in_tok=200, out_tok=80):
        self.content = content
        self.usage_metadata = {'input_tokens': in_tok, 'output_tokens': out_tok}
        self._model = model

REPLIES = {
    'price':   'iPhone 16 Pro (256GB): Rs 1,19,900. Diwali cashback: Rs 7,000 with HDFC.',
    'stock':   'In stock. Ships in 1-2 days. 847 units remaining.',
    'review':  'Excellent camera (4.7/5, 12,400 reviews). Battery improved 22%. Minor: slippery back.',
    'return':  'Electronics: 10-day return window. Original packaging required.',
    'compare': 'iPhone 16 Pro excels in video; Samsung S25 Ultra leads in zoom photography.',
    'gift':    'For a photography enthusiast: iPhone 16 Pro. Includes Diwali cashback offer.',
}

class MockLLM:
    def __init__(self, model='gpt-4o-mini', delay=0.05, fail_once=False):
        self.model = model
        self.delay = delay
        self._fail_next = fail_once
    def invoke(self, msg):
        text = msg if isinstance(msg, str) else str(msg)
        if self._fail_next:
            self._fail_next = False
            raise Exception('429 Too Many Requests')
        time.sleep(self.delay)
        for k, v in REPLIES.items():
            if k in text.lower(): return MockResponse(v, self.model)
        return MockResponse(f'[{self.model}] Answer for: {text[:40]}...', self.model)
    def stream(self, msg):
        r = self.invoke(msg)
        for w in r.content.split():
            time.sleep(0.02)
            yield type('C', (), {'content': w + ' '})()
    def batch(self, msgs, config=None):
        return [self.invoke(m) for m in msgs]

small_model  = MockLLM('gpt-4o-mini', delay=0.05)
big_model    = MockLLM('gpt-4o',      delay=0.12)
broken_model = MockLLM('gpt-4o-mini', delay=0.02, fail_once=True)

def timed(fn):
    t0 = time.perf_counter()
    r  = fn()
    return r, round(time.perf_counter() - t0, 4)

PRICE_PER_MILLION = {
    'gpt-4o-mini': {'in': 0.15,  'out': 0.60},
    'gpt-4o':      {'in': 2.50,  'out': 10.00},
}
def cost_of(response, model_name):
    u = response.usage_metadata
    p = PRICE_PER_MILLION[model_name]
    return (u['input_tokens']*p['in'] + u['output_tokens']*p['out']) / 1_000_000

print('Setup complete. small_model and big_model are ready.')

Setup complete. small_model and big_model are ready.


---
## Exercise 1 · Latency & Percentiles  ⏱ ~10 minutes

**Concept:** LLM latency is not symmetric — a few slow calls drag P99 far above the mean.
We measure P50 (median), P95 (1 in 20), and P99 (1 in 100) to find who gets hurt.

**What you need to do:**

1. Use `np.random.lognormal(mean=0.18, sigma=0.55, size=500)` to simulate Maxi's latency during the Diwali sale.
2. Compute and print P50, P95, and P99 using `np.percentile()`.
3. Count how many shoppers wait more than **3 seconds**. Print that as a percentage.
4. Run `timed(lambda: small_model.invoke('price of iPhone 16 Pro'))` five times. Print each elapsed time.

**Hint — the formula:**
```
P50  = np.percentile(samples, 50)
pct  = (samples > threshold).sum() / len(samples) * 100
```

**Expected output should look like:**
```
P50 = 1.18s   P95 = 2.84s   P99 = 4.02s
Shoppers waiting > 3s: 6.2%  (311 shoppers per 5,000)
```

In [7]:
# Exercise 1 — YOUR CODE HERE
# Step 1: simulate latency

# Step 2: compute percentiles

# Step 3: count shoppers over 3s

# Step 4: run 5 timed() calls


---
## Exercise 2 · Hot Product Cache  ⏱ ~12 minutes

**Concept:** The top 200 Diwali SKUs account for 85% of all queries.
Cache their summaries by `product_id`. Pre-warm before midnight. Invalidate on price/stock change.

**What you need to do:**

**Part A — Build the cache:**
1. Create a dict called `product_cache = {}`.
2. Write a function `get_summary(product_id)` that:
   - Returns the cached value if `product_id` is already in `product_cache`
   - Otherwise calls `small_model.invoke(f"Summarise {product_id} for Diwali shoppers")`, stores the result in the cache, and returns it

**Part B — Pre-warm:**
3. Given this list of top SKUs, call `get_summary()` on each one and print how long the pre-warm takes in total:
```python
TOP_SKUS = ["SKU-IPHONE16PRO", "SKU-SAMSUNG-S25", "SKU-DYSON-V15",
            "SKU-AIRPODS-PRO", "SKU-KINDLE-SCRIBE"]
```

**Part C — Serve at midnight:**
4. Run 15 queries for random SKUs from `TOP_SKUS`. Print the total time and confirm that 0 extra LLM calls were made.

**Part D — Flash sale invalidation:**
5. The iPhone 16 Pro price just dropped 30%. Remove it from `product_cache` using `.pop()`.
   Confirm the key is gone by printing `product_cache.get("SKU-IPHONE16PRO", "not in cache")`.

**Key fact to remember:** `dict.pop(key, None)` removes the key without raising an error if it doesn't exist.

In [3]:
# Exercise 2 — YOUR CODE HERE
# Part A: build the cache function

# Part B: pre-warm

# Part C: serve at midnight

# Part D: flash-sale invalidation


---
## Exercise 3 · Parallel Product Card  ⏱ ~10 minutes

**Concept:** Each ShopMax product card needs three pieces of data: price, stock, and reviews.
Running them sequentially takes 3× as long as running them in parallel.

**What you need to do:**

**Part A — Sequential (measure it):**
1. Call `timed()` on three separate `small_model.invoke()` calls:
   - `"price of SKU-IPHONE16PRO"`
   - `"stock of SKU-IPHONE16PRO"`
   - `"review summary of SKU-IPHONE16PRO"`
2. Add up the three times and print the sequential total.

**Part B — Parallel (using `concurrent.futures`):**
3. Use `ThreadPoolExecutor(max_workers=3)` to run all three calls at the same time.
4. Time the whole block and print the parallel total.
5. Print the speedup ratio: `sequential_time / parallel_time`.

**Hint:**
```python
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    futures = {k: pool.submit(small_model.invoke, v) for k, v in prompts.items()}
    results = {k: f.result() for k, f in futures.items()}
```

**Expected:** Parallel should be ~3× faster than sequential.

**Bonus (2 min):** Stream Maxi's recommendation for `"Compare iPhone 16 Pro vs Samsung S25"`.
Measure and print time-to-first-word separately from total time.

In [4]:
# Exercise 3 — YOUR CODE HERE
sku = 'SKU-IPHONE16PRO'

# Part A: sequential

# Part B: parallel

# Bonus: streaming


---
## Exercise 4 · Rate Limiter · Retry · Fallback  ⏱ ~10 minutes

**Concept:** At 5,000 rec/sec, unprotected LLM calls will hit rate limits and fail silently.
The three-layer shield: rate limit → retry → fallback.

**What you need to do:**

**Part A — Rate limiter:**
1. Create a simple rate limiter class (or copy the pattern below) that enforces 5 calls/second:
```python
class RateLimiter:
    def __init__(self, rps):
        self.min_interval = 1.0 / rps
        self._last = 0.0
    def acquire(self):
        wait = self.min_interval - (time.perf_counter() - self._last)
        if wait > 0: time.sleep(wait)
        self._last = time.perf_counter()
```
2. Create `limiter = RateLimiter(rps=5)`.

**Part B — Retry:**
3. Write `invoke_with_retry(llm, prompt, max_attempts=3)` that:
   - Calls `limiter.acquire()` before each attempt
   - Catches exceptions and waits `2 ** attempt` seconds before retrying
   - Raises the exception if all attempts are exhausted
   - Prints a warning on each retry

**Part C — Fallback:**
4. Write `safe_invoke(prompt)` that:
   - Tries `invoke_with_retry(broken_model, prompt, max_attempts=2)`
   - If that fails, falls back to `big_model.invoke(prompt)` and prints "using fallback"

5. Call `safe_invoke("price of iPhone 16 Pro")` and print the result.
   `broken_model` is already loaded in setup — it fails on the first call.

In [5]:
# Exercise 4 — YOUR CODE HERE
# Part A: rate limiter

# Part B: retry

# Part C: fallback

# Test it


---
## Exercise 5 · Little's Law + Cost Routing  ⏱ ~18 minutes

### Part A — Little's Law  ⏱ ~8 min

**The formula:**
> **Calls in flight = Arrival rate × Time per call × (1 − Cache hit rate)**

This tells you how many simultaneous LLM calls Maxi needs to handle at any moment.

**Q1.** Write `calls_in_flight(rate, seconds_per_call, cache_hit_rate)` using the formula above.

**Q2.** Write `total_cost(total_queries, cache_hit_rate, share_on_small)` that computes:
- LLM calls = `total_queries × (1 - cache_hit_rate)`
- Small model calls = `llm_calls × share_on_small` at $0.0004 each
- Big model calls = remaining, at $0.006 each
- Return total cost in USD

**Q3.** ShopMax's Diwali targets:
- Calls in flight ≤ 500
- Total cost for 2 hours ≤ $5,000
- Sale parameters: 5,000 rec/sec, 1.2s per call, 2 hours duration

Find values of `cache_hit_rate` and `share_on_small` that satisfy BOTH targets.
Try a few combinations and print whether each one passes or fails.

---

### Part B — Cost Routing  ⏱ ~10 min

**Q4.** Fill in the router below:

```python
SIMPLE   = {"RETURN_POLICY", "ORDER_STATUS", "SIZE_GUIDE"}
COMPLEX  = {"PRODUCT_COMPARE", "GIFT_RECOMMENDATION", "TECHNICAL_SPEC"}
```

Write `route_and_answer(text, category)` that:
- Uses `small_model` for SIMPLE categories
- Uses `big_model` for COMPLEX categories
- Calls `cost_of(response, model_name)` (already loaded in setup) to compute cost
- Returns `(answer_text, model_name, cost_in_usd)`

**Q5.** Run these 4 queries through your router and print the model used + cost for each:

| Category | Query |
|----------|-------|
| RETURN_POLICY | "Can I return an iPhone bought during Diwali?" |
| PRODUCT_COMPARE | "iPhone 16 Pro vs Samsung S25 for video editing?" |
| ORDER_STATUS | "Where is my order ORD-20241027-99123?" |
| GIFT_RECOMMENDATION | "Best Diwali gift under Rs 5,000 for a 65-year-old?" |

**Q6.** Project the cost for both routing strategies over the full 2-hour sale (5,000 rec/sec):
- No routing (all gpt-4o)
- With routing (80% mini, 20% full)

Print the savings in USD and as a percentage.

In [6]:
# Exercise 5 — YOUR CODE HERE

# Q1: calls_in_flight()

# Q2: total_cost()

# Q3: find parameters that meet both targets
total_queries = 5_000 * 2 * 3600

# Q4: router function

# Q5: run 4 queries
queries = [
    ('RETURN_POLICY',       'Can I return an iPhone bought during Diwali?'),
    ('PRODUCT_COMPARE',     'iPhone 16 Pro vs Samsung S25 for video editing?'),
    ('ORDER_STATUS',        'Where is my order ORD-20241027-99123?'),
    ('GIFT_RECOMMENDATION', 'Best Diwali gift under Rs 5,000 for a 65-year-old?'),
]

# Q6: project to 2-hour sale


---
## Quick Self-Check

Before wrapping up, answer these in plain English (no code needed):

1. Why is P99 more important than mean latency for a flash sale?
2. What happens if you forget to invalidate the cache when a flash-sale price drops?
3. Why does parallel product card assembly NOT help if all three lookups are sequential inside each one?
4. What is the purpose of `recursion_limit` in a LangGraph agent?
5. A shopper asks "What is the return policy?" — which model should Maxi use, and why?

---

*If you finish early: go back to Exercise 3 and try to implement the same parallel pattern using LangGraph's `StateGraph` with three nodes connected to a single merge node.*